In [1]:
import pandas as pd
import numpy as np

In [2]:
df=pd.read_excel(r"C:\Users\samru\OneDrive\Desktop\SIH_2025\AI-Pollution-Forecast-and-Policy-Dashboard\ML\Raw\CCR_DATA_2023_25\2025_Pusa, Delhi - IMD.xlsx",skiprows=16)

In [3]:
df

,From Date,To Date,PM2.5,PM10,NO,NO2,NOx,CO,Ozone,Benzene,...,RH,WS,WD,SR,BP,Gust,Variance,AT,Power,CO2
0,01-01-2025 00:00,02-01-2025 00:00,138.89,237.51,17.27,28.77,29.35,1.98,24.35,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,02-01-2025 00:00,03-01-2025 00:00,154.92,250.72,23.71,27.35,33.82,2.14,24.42,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,03-01-2025 00:00,04-01-2025 00:00,232.51,392.39,65.81,55.33,82.94,3.20,27.82,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,04-01-2025 00:00,05-01-2025 00:00,220.16,327.06,46.81,67.46,73.94,3.38,28.83,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,05-01-2025 00:00,06-01-2025 00:00,134.01,228.73,13.9,37.89,31.46,2.16,18.89,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
640,12-11-2025 00:00,13-11-2025 00:00,NaN,NaN,0,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
641,13-11-2025 00:00,14-11-2025 00:00,NaN,NaN,0,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
642,14-11-2025 00:00,15-11-2025 00:00,NaN,NaN,0,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
643,15-11-2025 00:00,16-11-2025 00:00,NaN,NaN,0,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [4]:
# ---------- 2. Remove duplicate rows and columns ----------
df = df.drop_duplicates().reset_index(drop=True)
df = df.loc[:, ~df.T.duplicated()]
print("Shape after removing duplicates:", df.shape)

Shape after removing duplicates: (645, 10)


In [5]:
# ---------- 3. Handle missing values (drop >70% NaN, impute median otherwise) ----------
nan_thresh = 0.7

# Drop columns with >70% missing
cols_to_drop = df.columns[df.isnull().mean() > nan_thresh]
df = df.drop(columns=cols_to_drop)
print(f"Dropped columns (>{int(nan_thresh*100)}% NaN): {cols_to_drop.tolist()}")

# Drop rows with >70% missing
rows_to_drop = df.index[df.isnull().mean(axis=1) > nan_thresh]
df = df.drop(index=rows_to_drop).reset_index(drop=True)
print(f"Dropped rows (>{int(nan_thresh*100)}% NaN):", len(rows_to_drop))

# Impute remaining missing values
num_cols = df.select_dtypes(include=[np.number]).columns
cat_cols = df.select_dtypes(exclude=[np.number]).columns

for col in num_cols:
    median_val = df[col].median()
    df[col] = df[col].fillna(median_val)

for col in cat_cols:
    if df[col].isnull().any():
        mode_val = df[col].mode(dropna=True)
        if not mode_val.empty:
            df[col] = df[col].fillna(mode_val[0])

print("Missing values after imputation:\n", df.isnull().sum())

Dropped columns (>70% NaN): ['Benzene']
Dropped rows (>70% NaN): 4
Missing values after imputation:
 From Date    0
To Date      0
PM2.5        0
PM10         0
NO           0
NO2          0
NOx          0
CO           0
Ozone        0
dtype: int64


In [6]:
# ---------- 4. Handle outliers using IQR (with 70% rule) ----------

def get_outlier_mask(series):
    q1 = series.quantile(0.25)
    q3 = series.quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    return (series < lower) | (series > upper)

# Handle columns: drop if >70% outliers, else replace with median
outlier_thresh = 0.7
cols_to_drop = []
for col in num_cols:
    outlier_mask = get_outlier_mask(df[col])
    outlier_fraction = outlier_mask.mean()
    if outlier_fraction > outlier_thresh:
        cols_to_drop.append(col)
    else:
        median_val = df[col].median()
        df.loc[outlier_mask, col] = median_val
if cols_to_drop:
    df = df.drop(columns=cols_to_drop)
    print(f"Dropped numeric columns (>{int(outlier_thresh*100)}% outliers): {cols_to_drop}")
    
# Handle rows: drop if >70% numeric columns are outliers in a given row
outlier_matrix = df[num_cols].apply(get_outlier_mask)
row_outlier_fraction = outlier_matrix.mean(axis=1)
rows_to_drop = df.index[row_outlier_fraction > outlier_thresh]
df = df.drop(index=rows_to_drop).reset_index(drop=True)
if len(rows_to_drop) > 0:
    print(f"Dropped rows (>{int(outlier_thresh*100)}% outliers): {len(rows_to_drop)}")


In [7]:

# ---------- 6. Final check ----------
print("Final shape:", df.shape)
print(df.head())

Final shape: (641, 9)
          From Date           To Date   PM2.5    PM10     NO    NO2    NOx  \
0  01-01-2025 00:00  02-01-2025 00:00  138.89  237.51  17.27  25.78  25.35   
1  02-01-2025 00:00  03-01-2025 00:00  154.92  250.72  23.71  25.78  25.35   
2  03-01-2025 00:00  04-01-2025 00:00  232.51  392.39  65.81  25.78  25.35   
3  04-01-2025 00:00  05-01-2025 00:00  220.16  327.06  46.81  25.78  25.35   
4  05-01-2025 00:00  06-01-2025 00:00  134.01  228.73   13.9  25.78  25.35   

     CO  Ozone  
0  1.31  33.53  
1  1.31  33.53  
2  1.31  33.53  
3  1.31  33.53  
4  1.31  33.53  


In [8]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
numerics = df.select_dtypes(include=[np.number]).columns
df[numerics] = scaler.fit_transform(df[numerics])

In [9]:
df

,From Date,To Date,PM2.5,PM10,NO,NO2,NOx,CO,Ozone
0,01-01-2025 00:00,02-01-2025 00:00,138.89,237.51,17.27,-3.552714e-15,3.552714e-15,-4.440892e-16,7.105427e-15
1,02-01-2025 00:00,03-01-2025 00:00,154.92,250.72,23.71,-3.552714e-15,3.552714e-15,-4.440892e-16,7.105427e-15
2,03-01-2025 00:00,04-01-2025 00:00,232.51,392.39,65.81,-3.552714e-15,3.552714e-15,-4.440892e-16,7.105427e-15
3,04-01-2025 00:00,05-01-2025 00:00,220.16,327.06,46.81,-3.552714e-15,3.552714e-15,-4.440892e-16,7.105427e-15
4,05-01-2025 00:00,06-01-2025 00:00,134.01,228.73,13.9,-3.552714e-15,3.552714e-15,-4.440892e-16,7.105427e-15
...,...,...,...,...,...,...,...,...,...
636,12-11-2025 00:00,13-11-2025 00:00,0,155.15,0,-3.552714e-15,3.552714e-15,-4.440892e-16,7.105427e-15
637,13-11-2025 00:00,14-11-2025 00:00,0,155.15,0,-3.552714e-15,3.552714e-15,-4.440892e-16,7.105427e-15
638,14-11-2025 00:00,15-11-2025 00:00,0,155.15,0,-3.552714e-15,3.552714e-15,-4.440892e-16,7.105427e-15
639,15-11-2025 00:00,16-11-2025 00:00,0,155.15,0,-3.552714e-15,3.552714e-15,-4.440892e-16,7.105427e-15


In [10]:
df.to_excel('PusaIMD2025.xlsx', index=False)